# Cirrhosis Stage Prediction - Baseline Models

**Objective**: Establish baseline performance with leak-free modeling pipeline before synthetic augmentation.

**Author**: Michael Udousoro  
**Date**: January 15, 2026

In [1]:
import numpy as np
import pandas as pd
import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

DATA_PATH = "../data/raw/cirrhosis.csv"
TARGET = "Stage"

print(f"Configuration set:")
print(f"- Random seed: {SEED}")
print(f"- Target variable: {TARGET}")

Configuration set:
- Random seed: 42
- Target variable: Stage


## 1. Data Loading

In [2]:
df = pd.read_csv(DATA_PATH)

# Normalize missing indicators
df = df.replace(["NA", "Na", "nan", "None", ""], np.nan)

# Drop ID if present
if "ID" in df.columns:
    df = df.drop(columns=["ID"])

print(f"Dataset loaded: {df.shape}")
print(f"\nFirst few rows:")
df.head()

Dataset loaded: (418, 19)

First few rows:


,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0


## 2. Train/Test Split

Defining features (X) and target (y) to prevent data leakage.

In [3]:
y = df[TARGET]
X = df.drop(columns=[TARGET])

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nMissing values per feature:")
print(X.isnull().sum()[X.isnull().sum() > 0])

Features: 18 columns, 418 samples

Target distribution:
Stage
3.0    155
4.0    144
2.0     92
1.0     21
Name: count, dtype: int64

Missing values per feature:
Drug             106
Ascites          106
Hepatomegaly     106
Spiders          106
Cholesterol      134
Copper           108
Alk_Phos         106
SGOT             106
Tryglicerides    136
Platelets         11
Prothrombin        2
dtype: int64


## 3. Preprocessing Pipeline

Building a leak-free preprocessing pipeline:
- **Numeric features**: KNN imputation (k=5) + standardization
- **Categorical features**: Mode imputation + one-hot encoding

**Critical**: Pipeline only fits on training data during cross-validation to prevent data leakage.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer

# Identify feature types
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

# Numeric pipeline: KNN imputation + scaling
numeric_transformer = Pipeline(steps=[
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# Categorical pipeline: mode imputation + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into preprocessing pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ],
    remainder="drop"
)

print("\n✓ Preprocessing pipeline created!")

Numeric features (11): ['N_Days', 'Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
Categorical features (7): ['Status', 'Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']

✓ Preprocessing pipeline created!


## 4. Baseline Models

Testing two classical ML algorithms:
- **Logistic Regression**: Linear baseline
- **Random Forest**: Non-linear ensemble baseline (500 trees)

These establish our performance benchmarks before synthetic data augmentation.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Define models
logreg = LogisticRegression(max_iter=3000, random_state=SEED)
rf = RandomForestClassifier(n_estimators=500, random_state=SEED)

# Create full pipelines (preprocessing + model)
logreg_model = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])
rf_model = Pipeline(steps=[("preprocess", preprocess), ("model", rf)])

print("Models created:")
print("  - Logistic Regression")
print("  - Random Forest (500 trees)")

Models created:
  - Logistic Regression
  - Random Forest (500 trees)


## 5. Evaluation Strategy

Using **5-fold stratified cross-validation**:
- Maintains class distribution in each fold
- Preprocessing fits only on training folds (no leakage)
- Reports mean ± std for robust estimates

**Metrics**: Accuracy and macro-averaged F1 score

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Metrics to track
scoring = {
    "acc": "accuracy",
    "f1_macro": make_scorer(f1_score, average="macro")
}

def evaluate(name, model):
    """Evaluate model using stratified cross-validation"""
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=False)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy:  {scores['test_acc'].mean():.3f} ± {scores['test_acc'].std():.3f}")
    print(f"F1-macro:  {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")
    
    return scores

print("Evaluation function ready!")

Evaluation function ready!


## 6. Baseline Results

Running experiments to establish baseline performance.

In [8]:
# Drop rows where target is missing (can't train on these)
df_clean = df.dropna(subset=[TARGET])

print(f"Original dataset: {df.shape[0]} samples")
print(f"After dropping missing targets: {df_clean.shape[0]} samples")
print(f"Dropped: {df.shape[0] - df_clean.shape[0]} rows with missing Stage\n")

# Now define X and y
y = df_clean[TARGET]
X = df_clean.drop(columns=[TARGET])

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nMissing values per feature:")
missing = X.isnull().sum()[X.isnull().sum() > 0]
if len(missing) > 0:
    print(missing)
else:
    print("No missing values in features!")

Original dataset: 418 samples
After dropping missing targets: 412 samples
Dropped: 6 rows with missing Stage

Features: 18 columns, 412 samples

Target distribution:
Stage
3.0    155
4.0    144
2.0     92
1.0     21
Name: count, dtype: int64

Missing values per feature:
Drug             100
Ascites          100
Hepatomegaly     100
Spiders          100
Cholesterol      128
Copper           102
Alk_Phos         100
SGOT             100
Tryglicerides    130
Platelets         11
Prothrombin        2
dtype: int64


## 3. Preprocessing Pipeline

Handling missing data and feature encoding without data leakage.

In [12]:
# Re-identify feature types
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("impute", KNNImputer(n_neighbors=5)),
    ("scale", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, num_features),
    ("categorical", categorical_pipeline, cat_features)
])

print(f"Numeric features: {len(num_features)}")
print(f"Categorical features: {len(cat_features)}")

Numeric features: 11
Categorical features: 7


## 4. Model Definition

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=3000, random_state=SEED))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("classifier", RandomForestClassifier(n_estimators=500, random_state=SEED))
    ])
}

print(f"Models defined: {list(models.keys())}")

Models defined: ['Logistic Regression', 'Random Forest']


## 5. Cross-Validation Setup

5-fold stratified cross-validation with accuracy and macro F1-score.

In [16]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": make_scorer(f1_score, average="macro")
}

def evaluate_model(model, X, y):
    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    return {
        "accuracy": (results["test_accuracy"].mean(), results["test_accuracy"].std()),
        "f1_macro": (results["test_f1_macro"].mean(), results["test_f1_macro"].std())
    }

## 6. Results

In [18]:
results = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    scores = evaluate_model(model, X, y)
    results[name] = scores
    
    print(f"Accuracy: {scores['accuracy'][0]:.3f} ± {scores['accuracy'][1]:.3f}")
    print(f"F1-macro: {scores['f1_macro'][0]:.3f} ± {scores['f1_macro'][1]:.3f}")


print("Baseline evaluation complete")



Evaluating Logistic Regression...
Accuracy: 0.495 ± 0.054
F1-macro: 0.352 ± 0.044

Evaluating Random Forest...


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ we

Accuracy: 0.517 ± 0.042
F1-macro: 0.356 ± 0.032
Baseline evaluation complete


/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaeludousoro/Desktop/Liver_Cirrohsis_Paper/env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
